In [ ]:
import dataclasses
import logging
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    _WAVELET_BAND_FREQ_RESOLUTION_HZ,
    _wavelet_transform,
    analyzers_to_datasets,
    load_analyzers,
)
from src.analysis.isc import compute_loo_isc  # noqa: E402
from src.analysis.mean_variance import compute_intersubject_stats  # noqa: E402
from src.definitions.constants import ExperimentNames, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Wavelet Power Exploration

Exploratory analysis of **wavelet-transformed EEG data** (power representation).

This notebook loads pre-computed (or freshly computed) wavelet transforms in 4-D
format `(n_subjects, n_channels, n_frequencies, n_times)` and runs the following
sketch analyses:

1. Grand-average spectral profile
2. Time–frequency map (spectrogram)
3. Per-band power time course
4. Intersubject variance in the wavelet domain
5. Wavelet-domain LOO-ISC (per band)

See `README.md` in this directory for the rationale behind each step and ideas
for future extensions.

## Configuration

In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / _WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Compute-only subject subset ──────────────────────────────────────────────
# When REUSE_WAVELETS=False, limit to the first N individuals to speed up
# interactive exploration.  Set to None to use all.  Ignored when loading cache.
N_SUBJECTS_SUBSET: int | None = 3

# ── Scope ─────────────────────────────────────────────────────────────────────
RUN_BROADBAND = True
RUN_PER_BAND = True

# ── Storage directory (same default as the CLI script) ────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.PROCESSED_DATA_DIR / ExperimentNames.PSILO_MUSIC.value / "wavelets"
)

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = ProjectPaths.NOTEBOOKS_DIR / "03-wavelet-analysis" / "plots" / "power"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"Reshape to 4D          : {RESHAPE_FREQUENCY_DIM}")
print(
    "Compute subset         : "
    + (
        f"{N_SUBJECTS_SUBSET} individuals"
        if not REUSE_WAVELETS and N_SUBJECTS_SUBSET is not None
        else "all individuals"
    )
)

## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
)
datasets = analyzers_to_datasets(analyzers)

# Optional subject subset when computing from scratch
if not REUSE_WAVELETS and N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals for compute (subset mode).")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, {ad.n_samples} samples"
    )

## Load or compute wavelet transforms

### Broadband

Stored in `WAVELET_DIR/broadband/`.

In [ ]:
broadband_datasets: dict = {}

if RUN_BROADBAND:
    broadband_datasets = _wavelet_transform(
        datasets=datasets,
        freqs=FREQS,
        representation=REPRESENTATION,
        keep_frequency_dim=KEEP_FREQUENCY_DIM,
        reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
        wavelet_dir=WAVELET_DIR / "broadband",
        reuse_wavelets=REUSE_WAVELETS,
    )
    for label, ad in broadband_datasets.items():
        source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
        print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

### Per-band

Each EEG band is stored in `WAVELET_DIR/band_<name>/`.

In [ ]:
band_datasets: dict[str, dict] = {}  # band_name -> {label: AnalysisData}

if RUN_PER_BAND:
    for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
        n_freqs = max(
            2,
            int(round((h_freq - l_freq) / _WAVELET_BAND_FREQ_RESOLUTION_HZ)) + 1,
        )
        band_freqs = np.linspace(l_freq, h_freq, n_freqs)
        band_datasets[band] = _wavelet_transform(
            datasets=datasets,
            freqs=band_freqs,
            representation=REPRESENTATION,
            keep_frequency_dim=KEEP_FREQUENCY_DIM,
            reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
            wavelet_dir=WAVELET_DIR / f"band_{band}",
            reuse_wavelets=REUSE_WAVELETS,
        )
        for label, ad in band_datasets[band].items():
            source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
            print(f"[{band:6s}] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use `bb_data`
(broadband, 4-D) and the derived scalars.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]
# LABEL = list(broadband_datasets.keys())[1]  # uncomment for second music type

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## 1 — Grand-Average Spectral Profile

Average wavelet power across subjects and time → `(n_channels, n_freqs)`;
then average over channels → `(n_freqs,)` spectrum.

Provides a sanity check: classical EEG peaks (alpha ~10 Hz) should be visible.

In [ ]:
# Grand average: mean over subjects & time → (n_channels, n_freqs)
grand_avg_ch_freq = bb_data.mean(axis=(0, 3))  # (n_channels, n_freqs)
# Channel average → (n_freqs,)
grand_avg_spectrum = grand_avg_ch_freq.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: spectrum
axes[0].plot(FREQS, grand_avg_spectrum, color="steelblue", linewidth=1.5)
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Mean wavelet power")
axes[0].set_title(f"Grand-average spectrum — {LABEL}")
# Shade canonical bands
band_colors = {
    "delta": "#d4e6f1",
    "theta": "#d5f5e3",
    "alpha": "#fdebd0",
    "beta": "#fadbd8",
    "gamma": "#e8daef",
}
for band, (lo, hi) in FREQUENCY_BANDS.items():
    axes[0].axvspan(lo, hi, alpha=0.25, color=band_colors.get(band, "grey"), label=band)
axes[0].legend(fontsize=8, loc="upper right")

# Panel B: channel × frequency image
im = axes[1].imshow(
    grand_avg_ch_freq,
    aspect="auto",
    origin="lower",
    extent=[FREQS[0], FREQS[-1], 0, n_channels],
    cmap="viridis",
)
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Channel index")
axes[1].set_title(f"Channel × frequency power — {LABEL}")
fig.colorbar(im, ax=axes[1], label="Power")

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "spectral_profile.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2 — Time–Frequency Map (Spectrogram)

Channel-averaged and subject-averaged wavelet power as a 2-D image
`(n_freqs × n_times)`.  Reveals how spectral content evolves over the stimulus.

In [ ]:
# Average over subjects and channels → (n_freqs, n_times)
tfr_map = bb_data.mean(axis=(0, 1))  # (n_freqs, n_times)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(
    tfr_map,
    aspect="auto",
    origin="lower",
    extent=[time[0], time[-1], FREQS[0], FREQS[-1]],
    cmap="inferno",
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title(f"Time–frequency map (mean wavelet power) — {LABEL}")
fig.colorbar(im, ax=ax, label="Power")

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "time_frequency_map.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3 — Per-Band Power Time Course

For each band, average wavelet power over the band's frequency bins, then compute
the channel-averaged group mean ± SD across subjects.  Directly comparable to the
broadband time series in notebook `01-*` but now frequency-resolved.

In [ ]:
fig, axes = plt.subplots(
    len(FREQUENCY_BANDS), 1, figsize=(14, 3 * len(FREQUENCY_BANDS)), sharex=True
)
if len(FREQUENCY_BANDS) == 1:
    axes = [axes]

for ax, (band, (lo, hi)) in zip(axes, FREQUENCY_BANDS.items()):
    # Select frequency indices within this band
    band_mask = (FREQS >= lo) & (FREQS <= hi)
    # Average over band freqs → (n_subjects, n_channels, n_times)
    band_power = bb_data[:, :, band_mask, :].mean(axis=2)
    # Channel average → (n_subjects, n_times)
    band_ch_avg = band_power.mean(axis=1)
    group_mean = band_ch_avg.mean(axis=0)
    group_std = band_ch_avg.std(axis=0)

    ax.plot(time, group_mean, color="steelblue", linewidth=0.8, label="Mean")
    ax.fill_between(
        time,
        group_mean - group_std,
        group_mean + group_std,
        alpha=0.25,
        color="steelblue",
        label="±1 SD",
    )
    ax.set_ylabel("Power")
    ax.set_title(f"{band} ({lo:.0f}–{hi:.0f} Hz)")
    ax.legend(fontsize=7, loc="upper right")

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Per-band wavelet power time course — {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "band_power_timecourse.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4 — Intersubject Variance in the Wavelet Domain

For each band, collapse frequencies to get `(n_subjects, n_channels, n_times)`,
then compute `compute_intersubject_stats` (same statistic as `01-*`).  Low
intersubject variance indicates frequency-resolved synchrony.

In [ ]:
fig, axes = plt.subplots(
    len(FREQUENCY_BANDS), 1, figsize=(14, 3 * len(FREQUENCY_BANDS)), sharex=True
)
if len(FREQUENCY_BANDS) == 1:
    axes = [axes]

for ax, (band, (lo, hi)) in zip(axes, FREQUENCY_BANDS.items()):
    band_mask = (FREQS >= lo) & (FREQS <= hi)
    band_power_3d = bb_data[:, :, band_mask, :].mean(axis=2)  # (n_sub, n_ch, n_t)
    stats = compute_intersubject_stats(band_power_3d)
    var_t = stats["var_t"]  # (n_times,)

    ax.plot(time, var_t, color="coral", linewidth=0.8)
    ax.set_ylabel("Variance")
    ax.set_title(f"{band} ({lo:.0f}–{hi:.0f} Hz) — intersubject variance")

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Wavelet-domain intersubject variance — {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "wavelet_intersubject_variance.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()

---
## 5 — Wavelet-Domain LOO-ISC (Per Band)

For each band, compute LOO-ISC on band-averaged power
`(n_subjects, n_channels, n_times)` — direct counterpart of `02-*` ISC but in
the wavelet domain.  We expect stimulus-tracking bands (e.g. delta/theta for
rhythm) to show higher ISC.

In [ ]:
band_mean_iscs: dict[str, np.ndarray] = {}

for band, (lo, hi) in FREQUENCY_BANDS.items():
    band_mask = (FREQS >= lo) & (FREQS <= hi)
    band_power_3d = bb_data[:, :, band_mask, :].mean(axis=2)
    _, mean_loo_isc = compute_loo_isc(band_power_3d)
    band_mean_iscs[band] = mean_loo_isc  # (n_channels,)
    print(
        f"  {band:6s}  mean LOO-ISC = {mean_loo_isc.mean():.4f}"
        f"  (median = {np.median(mean_loo_isc):.4f})"
    )

In [ ]:
# Bar chart of per-band mean ISC
bands_list = list(band_mean_iscs.keys())
means = [band_mean_iscs[b].mean() for b in bands_list]
medians = [np.median(band_mean_iscs[b]) for b in bands_list]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(bands_list))
width = 0.35
ax.bar(x - width / 2, means, width, label="Mean", color="steelblue")
ax.bar(x + width / 2, medians, width, label="Median", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(bands_list)
ax.set_ylabel("LOO-ISC (channel average)")
ax.set_title(f"Per-band wavelet LOO-ISC — {LABEL}")
ax.legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "wavelet_loo_isc_bar.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.1 — LOO-ISC distribution per band

Histogram of per-channel mean LOO-ISC for each band, clipped at the 99th
percentile to suppress outlier channels.

In [ ]:
fig, axes = plt.subplots(
    1, len(FREQUENCY_BANDS), figsize=(3.5 * len(FREQUENCY_BANDS), 4), sharey=True
)
if len(FREQUENCY_BANDS) == 1:
    axes = [axes]

for ax, band in zip(axes, FREQUENCY_BANDS):
    vals = band_mean_iscs[band]
    clip = np.percentile(vals, 99)
    ax.hist(
        vals[vals <= clip], bins=30, color="steelblue", edgecolor="white", alpha=0.8
    )
    ax.axvline(
        vals.mean(),
        color="red",
        linestyle="--",
        linewidth=1,
        label=f"mean={vals.mean():.3f}",
    )
    ax.set_xlabel("LOO-ISC")
    ax.set_title(band)
    ax.legend(fontsize=7)

axes[0].set_ylabel("Number of channels")
fig.suptitle(f"Wavelet LOO-ISC distributions — {LABEL}", fontsize=13, y=1.02)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "wavelet_loo_isc_distributions.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()

---
## Summary

The arrays are available for further analysis:

- `broadband_datasets[LABEL].data` — 4-D broadband wavelet power
- `band_datasets[band][LABEL].data` — 4-D per-band wavelet power
- `band_mean_iscs[band]` — per-channel LOO-ISC for each band

See `README.md` in this directory for ideas on extending these analyses
(phase ISC, condition comparison, topographic mapping, etc.).